# Score the eight datasets

Scores the context mean, PIE and PWPA with PIE's metric code (FDR 0.05) on every development fold, then takes the mean and the standard error over folds.
Writes `results/eight_datasets_scores.tsv` and draws `figures/eight_datasets.png`.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../utils")
from paths import data_dir

In [ ]:
# TODO(orcd): port immune_grn/lab/exp/K062-pie-eight/score_all.py and collect.py. Until then
# results/eight_datasets_scores.tsv holds K062's scores; its PWPA rows come from gates fitted on PIE's five sources
# without RFE, and rerunning 2.train_pwpa.ipynb will change them.
#
# rows = []
# for dataset, model, fold: rows += PIE's metrics of the predictions against the truth
# scores = rows.groupby(["dataset", "model", "metric"]).value: mean, and se = sd / sqrt(number of folds)
# scores.to_csv("results/eight_datasets_scores.tsv", sep="\t", index=False)

In [ ]:
import matplotlib.pyplot as plt

scores = pd.read_csv("results/eight_datasets_scores.tsv", sep="\t")
DATASETS = {
    "replogle_k562": "Replogle 2022\nK562",
    "replogle_rpe1": "Replogle 2022\nRPE1",
    "nadig_hepg2": "Nadig 2025\nHepG2",
    "nadig_jurkat": "Nadig 2025\nJurkat, resting",
    "kolf_ipsc": "Nourreddine 2026 ‡\nKOLF2.1J iPSC",
    "vcc_h1": "Roohani 2025\nH1 hESC (VCC)",
    "shi_brain": "Shi 2026 *\nmouse brain",
    "geiger_bmdc": "Geiger-Schuller 2026 †\nmouse dendritic cells",
}
METRICS = {
    "binary_auprc": "AUPRC",
    "sig_jaccard": "Jaccard",
    "direction_match": "DE direction match",
    "spearman_nsig": "Spearman effect size",
    "spearman_lfc": "Spearman LFC",
    "discrimination_score_l1": "Discrimination L1",
}
MODELS = {"context_mean": ("Context mean", "#8f96a3"), "pie": ("PIE, trained on each dataset", "#3b68a3"), "pwpa": ("PWPA", "#c98fc4")}

fig, axes = plt.subplots(len(DATASETS), len(METRICS), figsize=(15, 2 * len(DATASETS)), sharey="col")
for r, (dataset, label) in enumerate(DATASETS.items()):
    for c, metric in enumerate(METRICS):
        ax = axes[r, c]
        cell = scores.query("dataset == @dataset and metric == @metric").set_index("model")
        for x, (model, (_, color)) in enumerate(MODELS.items()):
            mean, se = cell.loc[model, ["mean", "se"]]
            if pd.isna(mean):  # the context mean has no Spearman effect size
                continue
            ax.bar(x, mean, yerr=se, color=color, width=0.75, error_kw={"lw": 0.8, "capsize": 2})
            ax.text(x, max(mean, 0) + (0 if pd.isna(se) else se), f"{mean:.2f}", ha="center", va="bottom", fontsize=7)
        ax.axhline(0, color="black", lw=0.5)
        ax.set_xticks([])
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        if r == 0:
            ax.set_title(METRICS[metric], fontsize=10)
        if c == 0:
            ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
handles = [plt.Rectangle((0, 0), 1, 1, color=color) for _, color in MODELS.values()]
fig.legend(handles, [name for name, _ in MODELS.values()], loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig("figures/eight_datasets.png", dpi=200)

\* Shi: in vivo data from 74 mice. The labels use PIE's cell-level DE, which treats each cell as a replicate and ignores mouse-to-mouse variation.

† Geiger-Schuller: primary mouse dendritic cells in 16 hashtagged samples over 2 rounds. Cell-level DE ignores this sample structure; a pseudobulk test across samples may label differently.

‡ Nourreddine (KOLF): each knockdown was screened in one of three batches, but cell-level DE compares it with non-targeting cells pooled across all batches (PIE's default), so batch differences can enter the DE labels.